In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score



In [ ]:
DATA = Path("../data")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)



In [ ]:
# ======================
# 1. ЗАГРУЗКА ДАННЫХ
# ======================
df = pd.read_csv(DATA / "toxic_comments_sample.csv")



In [ ]:
print("Number of comments:", df.shape[0])
print("Columns:", list(df.columns))



In [ ]:
df.info()
df.head()



In [ ]:
# Подготовка данных: баланс токсичных / чистых комментариев
class_counts = df["toxic"].value_counts().sort_index()
labels_map = {0: "clean", 1: "toxic"}
display_idx = [labels_map[i] for i in class_counts.index]
percentages = (class_counts / class_counts.sum()) * 100

plt.figure(figsize=(7, 5))
colors = sns.color_palette("viridis", len(class_counts))
ax = plt.subplot(111)

wedges, _ = ax.pie(
    class_counts,
    startangle=90,
    colors=colors,
    wedgeprops=dict(width=0.5, edgecolor="w", linewidth=0.5),
)

legend_labels = [
    f"{name} ({count} - {pct:.1f}%)"
    for name, count, pct in zip(display_idx, class_counts, percentages)
]
plt.legend(wedges, legend_labels, title="Class", loc="center left", bbox_to_anchor=(1, 0.5))
plt.title("Toxic vs clean comments", fontsize=14, fontweight="bold", pad=16)

centre = plt.Circle((0, 0), 0.3, color="white")
ax.add_artist(centre)
plt.text(0, 0, f"Total:\n{class_counts.sum()}", ha="center", va="center")

plt.tight_layout()
plt.savefig(FIG / "toxic_class_balance.png", dpi=120, bbox_inches="tight")
plt.show()



In [ ]:
# Столбчатая диаграмма баланса
plt.figure(figsize=(7, 4))
sns.barplot(
    x=display_idx,
    y=class_counts.values,
    hue=display_idx,
    palette="viridis",
    legend=False,
)
plt.title("Class balance", fontsize=14, pad=12)
plt.ylabel("Count")
plt.grid(axis="y", linestyle="--", alpha=0.7)
for i, c in enumerate(class_counts.values):
    plt.text(i, c + 1, str(c), ha="center")
plt.tight_layout()
plt.show()



In [ ]:
# Длина текстов — полезный sanity-check для NLP
df = df.copy()
df["n_chars"] = df["text"].astype(str).str.len()
df["n_tokens"] = df["text"].astype(str).str.split().str.len()

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(data=df, x="n_chars", hue="toxic", bins=20, palette="viridis")
plt.title("Comment length (chars)")
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="toxic", y="n_tokens", palette="viridis")
plt.title("Tokens by class")
plt.tight_layout()
plt.show()



In [ ]:
# ======================
# 2. TRAIN / TEST SPLIT
# ======================
RANDOM_SEED = 0
X_train, X_test, y_train, y_test = train_test_split(
    df["text"].astype(str),
    df["toxic"],
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=df["toxic"],
)
print("Train:", len(X_train), "Test:", len(X_test))



In [ ]:
# ======================
# 3. TF-IDF ВЕКТОРИЗАЦИЯ
# ======================
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),  # униграммы + биграммы
    min_df=1,
    max_features=5000,
)
X_train_v = vectorizer.fit_transform(X_train)
X_test_v = vectorizer.transform(X_test)
print("TF-IDF shape train:", X_train_v.shape)



In [ ]:
# ======================
# 4. ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ
# ======================
model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
model.fit(X_train_v, y_train)
pred = model.predict(X_test_v)
print("accuracy:", round(accuracy_score(y_test, pred), 4))



In [ ]:
print(classification_report(y_test, pred, digits=3, target_names=["clean", "toxic"]))



In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["clean", "toxic"],
    yticklabels=["clean", "toxic"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()



In [ ]:
# Топ весов модели — слова, связанные с toxic=1
coef = model.coef_[0]
names = vectorizer.get_feature_names_out()
top = sorted(zip(coef, names), reverse=True)[:15]
bottom = sorted(zip(coef, names))[:15]

print("Top toxic terms:")
for w, t in top:
    print(f"  {t:20s} {w:.3f}")
print("Top clean terms:")
for w, t in bottom:
    print(f"  {t:20s} {w:.3f}")



In [ ]:
# Визуализация топ токсичных терминов
top_df = pd.DataFrame(top, columns=["coef", "term"]).iloc[::-1]

plt.figure(figsize=(8, 6))
plt.barh(top_df["term"], top_df["coef"], color=sns.color_palette("viridis", len(top_df)))
plt.title("Top toxic TF-IDF coefficients", fontsize=14, pad=12)
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()



In [ ]:
# Примеры предсказаний
sample = pd.DataFrame({"text": X_test.values, "true": y_test.values, "pred": pred})
sample.head(10)

